In [0]:
# Databricks Notebook: Data Objects Proof
# Proving Managed Table, External Table, View, and Materialized View

# --- SETUP ---
CATALOG_NAME = "training_medallion" # Adjust to your catalog
SCHEMA_NAME = f"{CATALOG_NAME}.00_landing"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")

print("--- 1. MANAGED TABLE ---")
# Unity Catalog fully manages lifecycle and storage
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SCHEMA_NAME}.gold_talent_metrics (
        employee_id INT, performance_score INT, status STRING
    ) USING DELTA
""")
spark.sql(f"INSERT INTO {SCHEMA_NAME}.gold_talent_metrics VALUES (1, 95, 'Active'), (2, 80, 'Inactive')")

# Evidence: Table Type is MANAGED and location is inside Unity Catalog managed storage
display(spark.sql(f"DESCRIBE EXTENDED {SCHEMA_NAME}.gold_talent_metrics").filter("col_name IN ('Type', 'Location')"))

print("\n--- 2. EXTERNAL TABLE ---")
# --- READING A FILE FROM A VOLUME AS A VIEW ---
VOLUME_PATH = "/Volumes/training_medallion/00_landing/employee_documents"

# 1. Ensure the Volume exists and put the CSV file into it
spark.sql("CREATE VOLUME IF NOT EXISTS training_medallion.00_landing.employee_documents")
dbutils.fs.put(f"{VOLUME_PATH}/legacy_risk_data.csv", "id,risk_level\n101,High\n102,Low", True)

# 2. Create a VIEW that queries the CSV file inside the Volume directly
spark.sql(f"""
    CREATE OR REPLACE VIEW training_medallion.00_landing.legacy_incident_logs AS 
    SELECT * FROM csv.`{VOLUME_PATH}/legacy_risk_data.csv`
""")

# 3. Evidence: Display the data as if it were a standard table
print("-> Evidence: Successfully queried tabular data directly from a raw file inside the Volume.")
display(spark.sql("SELECT * FROM training_medallion.00_landing.legacy_incident_logs_v"))

# 4. Prove that this object is a View, not a Table
display(spark.sql("DESCRIBE EXTENDED training_medallion.00_landing.legacy_incident_logs").filter("col_name = 'Type'"))

print("\n--- 3. VIEW ---")
# Stores only the query definition (virtual), does not duplicate data
spark.sql(f"""
    CREATE OR REPLACE VIEW {SCHEMA_NAME}.active_employees_v AS 
    SELECT employee_id, performance_score 
    FROM {SCHEMA_NAME}.gold_talent_metrics 
    WHERE status = 'Active'
""")

# Evidence: Type is VIEW, executed on-the-fly
display(spark.sql(f"DESCRIBE EXTENDED {SCHEMA_NAME}.active_employees_v").filter("col_name = 'Type'"))

print("\n--- 4. MATERIALIZED VIEW ---")
# Stores precomputed query results for fast dashboards
try:
    spark.sql(f"""
        CREATE MATERIALIZED VIEW IF NOT EXISTS {SCHEMA_NAME}.monthly_risk_summary AS
        SELECT risk_level, COUNT(id) as total_incidents
        FROM {SCHEMA_NAME}.legacy_incident_logs
        GROUP BY risk_level
    """)
    # Evidence: Type is MATERIALIZED VIEW
    display(spark.sql(f"DESCRIBE EXTENDED {SCHEMA_NAME}.monthly_risk_summary").filter("col_name = 'Type'"))
except Exception as e:
    print(f"-> Info: Materialized View creation requires a Databricks SQL Warehouse or Serverless compute. (Error log: {e})")

--- 1. MANAGED TABLE ---


col_name,data_type,comment
Type,MANAGED,
Location,,



--- 2. EXTERNAL TABLE ---
Wrote 30 bytes.
-> Evidence: Successfully queried tabular data directly from a raw file inside the Volume.


_c0,_c1
id,risk_level
101,High
102,Low


col_name,data_type,comment
Type,VIEW,



--- 3. VIEW ---


col_name,data_type,comment
Type,VIEW,



--- 4. MATERIALIZED VIEW ---
-> Info: Materialized View creation requires a Databricks SQL Warehouse or Serverless compute. (Error log: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `risk_level` cannot be resolved. Did you mean one of the following? [`_c0`, `_c1`]. SQLSTATE: 42703

JVM stacktrace:
com.databricks.pipelines.common.errors.formatting.FormattedAnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.unresolvedAttributeError(QueryCompilationErrors.scala:962)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.org$apache$spark$sql$catalyst$analysis$CheckAnalysis$$failUnresolvedAttribute(CheckAnalysis.scala:301)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$16(CheckAnalysis.scala:719)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$16$adapted(CheckAnalysis.scala:704)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.traverse$1(CheckAnalysi